# Demo 01 : State schemas

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · typed-dict

The project board of Comet Arcade is one dictionary. The Designer returns a partial update : one key. The other keys keep their values.

In [2]:
import json
from dataclasses import dataclass
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

with open(data("game-studio/kit/briefs.json"), encoding="utf-8") as file:
    brief = json.load(file)[0]["brief"]

class Board(TypedDict):
    brief: str
    rules: str
    code: str
    bugs: list[str]
    attempts: int

def designer(board: Board):
    return {"rules": "Eat stars. Dodge comets."}

builder = StateGraph(Board)
builder.add_node("designer", designer)
builder.add_edge(START, "designer")
builder.add_edge("designer", END)
inputs = {"brief": brief, "rules": "", "code": "", "bugs": [], "attempts": 0}
print("typed dict :", builder.compile().invoke(inputs))

typed dict : {'brief': 'Playtesters love Comet Snake but rage-quit when one comet ends a long run. Give them one way to survive.', 'rules': 'Eat stars. Dodge comets.', 'code': '', 'bugs': [], 'attempts': 0}


We use a dataclass when state needs defaults. Nodes receive an instance and still return a dictionary update.

In [3]:
@dataclass
class BoardDefaults:
    brief: str
    attempts: int = 0

def count_attempt(board: BoardDefaults):
    print("default :", board.attempts)
    return {"attempts": board.attempts + 1}

defaults = StateGraph(BoardDefaults)
defaults.add_node("count_attempt", count_attempt)
defaults.add_edge(START, "count_attempt")
defaults.add_edge("count_attempt", END)
print("dataclass :", defaults.compile().invoke({"brief": brief}))

default : 0
dataclass : {'brief': 'Playtesters love Comet Snake but rage-quit when one comet ends a long run. Give them one way to survive.', 'attempts': 1}


## b · pydantic

We validate the input to a node with Pydantic. The crew has a cap of 3 attempts. We catch the invalid input so Run All continues.

In [4]:
from pydantic import BaseModel, Field, ValidationError

class CheckedBoard(BaseModel):
    brief: str
    attempts: int = Field(ge=0, le=3)

def next_attempt(board: CheckedBoard):
    return {"attempts": board.attempts + 1}

validated = StateGraph(CheckedBoard)
validated.add_node("next_attempt", next_attempt)
validated.add_edge(START, "next_attempt")
validated.add_edge("next_attempt", END)
validated_graph = validated.compile()
print("valid :", validated_graph.invoke({"brief": brief, "attempts": 1}))
try:
    validated_graph.invoke({"brief": brief, "attempts": 9})
except ValidationError as error:
    print("invalid :", error.errors()[0]["type"])

valid : {'brief': 'Playtesters love Comet Snake but rage-quit when one comet ends a long run. Give them one way to survive.', 'attempts': 2}
invalid : less_than_equal


We write an invalid value on purpose to find where validation happens. Pydantic validates node inputs, so the bad write fails when a next node reads it. A bad final write has no next node, and `invoke` returns a plain dictionary.

In [5]:
boundary = StateGraph(CheckedBoard)
boundary.add_node("bad_write", lambda board: {"attempts": 9})
boundary.add_edge(START, "bad_write")
boundary.add_edge("bad_write", END)
print("final write :", boundary.compile().invoke({"brief": brief, "attempts": 1}))

followed = StateGraph(CheckedBoard)
followed.add_node("bad_write", lambda board: {"attempts": 9})
followed.add_node("reader", lambda board: {})
followed.add_edge(START, "bad_write")
followed.add_edge("bad_write", "reader")
followed.add_edge("reader", END)
try:
    followed.compile().invoke({"brief": brief, "attempts": 1})
except ValidationError as error:
    print("next node :", error.errors()[0]["type"])

final write : {'brief': 'Playtesters love Comet Snake but rage-quit when one comet ends a long run. Give them one way to survive.', 'attempts': 9}
next node : less_than_equal


## c · input-and-output

We accept a brief as input and return only the verdict. The internal board holds the code as well.

In [6]:
class BriefIn(TypedDict):
    brief: str

class VerdictOut(TypedDict):
    verdict: str

class InternalBoard(TypedDict):
    brief: str
    code: str
    verdict: str

public = StateGraph(InternalBoard, input_schema=BriefIn, output_schema=VerdictOut)
public.add_node("coder", lambda board: {"code": "function step() {}"})
public.add_node("tester", lambda board: {"verdict": "pass"})
public.add_edge(START, "coder")
public.add_edge("coder", "tester")
public.add_edge("tester", END)
print("public output :", public.compile().invoke({"brief": brief}))

public output : {'verdict': 'pass'}


The filter applies to the output of `invoke`. A checkpointer still saves the keys the output leaves out.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

saved_public = public.compile(checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "filtered"}}
print("invoke :", saved_public.invoke({"brief": brief}, thread))
print("checkpoint keys :", sorted(saved_public.get_state(thread).values))

invoke : {'verdict': 'pass'}
checkpoint keys : ['brief', 'code', 'verdict']


## d · private-state

The Coder leaves a note for the Tester in a private key, declared through a node's input schema. It is absent from `invoke` output but remains visible in state streams.

In [8]:
class Scratch(TypedDict):
    scratch: str

def coder(board: BriefIn):
    return {"scratch": "self-hit check rewritten"}

def tester(board: Scratch):
    return {"verdict": "retest : " + board["scratch"]}

private = StateGraph(InternalBoard, input_schema=BriefIn, output_schema=VerdictOut)
private.add_node("coder", coder)
private.add_node("tester", tester)
private.add_edge(START, "coder")
private.add_edge("coder", "tester")
private.add_edge("tester", END)
private_graph = private.compile()
short = {"brief": "one way to survive"}
print("output :", private_graph.invoke(short))
for part in private_graph.stream(short, stream_mode="values"):
    print("state :", part)

output : {'verdict': 'retest : self-hit check rewritten'}
state : {'brief': 'one way to survive'}
state : {'brief': 'one way to survive', 'scratch': 'self-hit check rewritten'}
state : {'brief': 'one way to survive', 'verdict': 'retest : self-hit check rewritten', 'scratch': 'self-hit check rewritten'}


## Things to try

Optional : we can change a brief, a thread id or a returned update and compare the saved state.